In [2]:
# ============================================================
# 🚀 PIPELINE FINAL: METABOLISMO + CLÍNICO + KM (VERSIÓN 2026)
# ============================================================

# 1. CARGA DE LIBRERÍAS
required_libs <- c("dplyr", "ggplot2", "reshape2", "ggpubr", "survival",
                   "survminer", "readxl", "purrr", "tidyr", "grid", "gridExtra",
                   "scales")
invisible(lapply(required_libs, library, character.only = TRUE))

# ===============================
# 2. CONFIGURACIÓN DE RUTAS
# ===============================
paths <- list(
  metabolic    = "/Users/eduardoruiz/Documents/GitHub/Precision-Oncology-for-Breast-Cancer-Diagnosis/src/Clustering and data analysis PYTHON/Metabolic data analysis/ML models using metabolic data/resultados_normas_pareto_PCA+UMAP/Merged_TCGA_BRCA_AllData_safe_withSelectedClusters_NormasPareto_cleaned.csv",
  clinicaldata = "/Users/eduardoruiz/Documents/GitHub/Precision-Oncology-for-Breast-Cancer-Diagnosis/src/Clustering and data analysis PYTHON/Clinical data analysis/ML models using clinical data/Results_clustering_UMAP/DATA_MASTER_CLUSTERS_Y_CLINICA.csv",
  div          = "/Users/eduardoruiz/Documents/GitHub/Precision-Oncology-for-Breast-Cancer-Diagnosis/src/Clustering and data analysis PYTHON/Cluster_correlations/pacientes_divergentes_para_estudio_pyn.csv",
  core         = "/Users/eduardoruiz/Documents/GitHub/Precision-Oncology-for-Breast-Cancer-Diagnosis/src/Clustering and data analysis PYTHON/Cluster_correlations/pacientes_core_correlacion_Paretoynormas.csv"
)

group_colors <- c("Core" = "#0072B2", "Divergent" = "#D55E00")

# ─── TEMA GLOBAL: tipografía grande en todos los plots ─────────────────────
base_theme <- theme_pubr() +
  theme(
    title         = element_text(size = 18, face = "bold"),
    axis.title    = element_text(size = 15),
    axis.text     = element_text(size = 13),
    legend.text   = element_text(size = 13),
    legend.title  = element_text(size = 14, face = "bold"),
    strip.text    = element_text(size = 14, face = "bold"),
    plot.subtitle = element_text(size = 13, color = "grey50")
  )

# ===============================
# 3. PREPROCESAMIENTO Y CRUCE
# ===============================
clean_names <- function(x) toupper(gsub("\\.", "-", trimws(as.character(x))))

df_metab <- read.csv(paths$metabolic, stringsAsFactors = FALSE)
df_clin  <- read.csv(paths$clinicaldata, stringsAsFactors = FALSE)
df_div   <- read.csv(paths$div, stringsAsFactors = FALSE)
df_core  <- read.csv(paths$core, stringsAsFactors = FALSE)

df_metab$ModelName <- clean_names(df_metab$ModelName)
df_clin$ModelName  <- clean_names(df_clin$ModelName)
div_names          <- clean_names(df_div$ModelName)
core_names         <- clean_names(df_core$ModelName)

# ─── FIX 1: Detectar colisiones ────────────────────────────────────────────
overlap <- intersect(div_names, core_names)
if (length(overlap) > 0) {
  warning(paste0("⚠️  ", length(overlap), " pacientes en ambos grupos (se excluirán): ",
                 paste(head(overlap, 5), collapse = ", ")))
}

analysis <- df_metab %>%
  mutate(Group = case_when(
    ModelName %in% overlap    ~ NA_character_,
    ModelName %in% div_names  ~ "Divergent",
    ModelName %in% core_names ~ "Core",
    TRUE                      ~ "Other"
  )) %>%
  filter(Group %in% c("Core", "Divergent"))

# ─── FIX 2: Evitar conflictos de nombre en el join ────────────────────────
clinical_cols <- c("ModelName", "menopause_status.diagnoses", "primary_diagnosis.diagnoses",
                   "ER", "PR", "HER2", "Subtype", "age_at_index.demographic", "OS", "OS.time")

clin_subset <- df_clin %>%
  select(any_of(clinical_cols)) %>%
  rename_with(~ paste0("clin_", .), any_of(c("OS", "OS.time")))

analysis <- analysis %>%
  select(-any_of(c("OS", "OS.time"))) %>%     # borrar originales de df_metab
  left_join(clin_subset, by = "ModelName") %>%
  # ─── FIX 3: OS puede ser NA ────────────────────────────────────────────
  mutate(OS = if_else(
    is.na(clin_OS), NA_real_,
    if_else(tolower(as.character(clin_OS)) %in% c("alive", "0"), 0, 1)
  )) %>%
  rename("OS.time" = "clin_OS.time") %>%
  select(-clin_OS)

# ===============================
# 4. ANÁLISIS ESTADÍSTICO (FDR)
# ===============================
run_stats <- function(data, columns) {
  results <- map_df(columns, function(col) {
    sub_data <- data %>%
      select(Group, value = !!sym(col)) %>%
      filter(is.finite(value), !is.na(value))

    group_sizes <- table(sub_data$Group)
    if (length(group_sizes) < 2 || any(group_sizes < 3)) return(NULL)

    test <- wilcox.test(value ~ Group, data = sub_data, exact = FALSE)

    # ─── FIX 4: Medianas por nombre, no por posición ───────────────────
    meds <- sub_data %>%
      group_by(Group) %>%
      summarise(m = median(value), .groups = "drop") %>%
      arrange(Group)

    data.frame(
      Feature    = col,
      Med_Core   = meds$m[meds$Group == "Core"],
      Med_Div    = meds$m[meds$Group == "Divergent"],
      p_val      = test$p.value
    )
  })

  if (nrow(results) == 0) return(results)

  results %>%
    mutate(p_adj = p.adjust(p_val, method = "fdr")) %>%
    arrange(p_val)
}

metab_res  <- run_stats(analysis, grep("SA_|Ratio", colnames(analysis), value = TRUE))
pareto_res <- run_stats(analysis, grep("^Pareto_", colnames(analysis), value = TRUE))

# ===============================
# 5. FUNCIONES DE PLOTEO
# ===============================

# --- Violin + Boxplot ---
plot_box_feat <- function(feat, data, stats_df) {
  # ─── FIX 5: Verificar que el feature existe en stats_df ───────────────
  p_info <- stats_df %>% filter(Feature == feat)
  if (nrow(p_info) == 0) return(NULL)
  p_info <- p_info %>% slice(1)

  plot_data <- data %>% filter(is.finite(!!sym(feat)), !is.na(!!sym(feat)))
  if (nrow(plot_data) == 0) return(NULL)

  ggplot(plot_data, aes(x = Group, y = !!sym(feat), fill = Group)) +
    geom_violin(alpha = 0.2, color = NA) +
    geom_boxplot(width = 0.4, outlier.shape = 16, outlier.size = 2) +
    labs(title = feat, subtitle = paste("p-adj:", signif(p_info$p_adj, 3)), y = NULL) +
    scale_fill_manual(values = group_colors) +
    base_theme +
    theme(legend.position = "none")
}

# --- Variables clínicas ---
plot_clinical <- function(var, data) {
  if (!var %in% colnames(data)) return(NULL)
  tmp <- data %>% filter(!is.na(!!sym(var)))
  if (nrow(tmp) == 0) return(NULL)

  if (is.numeric(tmp[[var]])) {
    ggplot(tmp, aes(x = Group, y = !!sym(var), fill = Group)) +
      geom_boxplot(outlier.shape = 16, outlier.size = 2) +
      stat_compare_means(label = "p.signif", label.size = 5) +
      scale_fill_manual(values = group_colors) +
      labs(title = paste("Analisis de", var), x = "") +
      base_theme
  } else {
    ggplot(tmp, aes(x = !!sym(var), fill = Group)) +
      geom_bar(position = "fill") +
      scale_y_continuous(labels = percent) +
      scale_fill_manual(values = group_colors) +
      labs(title = paste("Proporcion de", var), y = "Porcentaje") +
      base_theme +
      theme(axis.text.x = element_text(angle = 35, hjust = 1, size = 12))
  }
}

# ===============================
# 6. GENERACIÓN DEL REPORTE PDF
# ===============================
pdf("Reporte_Metabolismo_Clinico_2026.pdf", width = 14, height = 11)

# ─── Metabolismo ──────────────────────────────────────────────────────────
if (nrow(metab_res) > 0) {
  metab_plots <- map(head(metab_res$Feature, 6), ~plot_box_feat(.x, analysis, metab_res)) %>%
    Filter(Negate(is.null), .)
  if (length(metab_plots) > 0)
    grid.arrange(grobs = metab_plots, ncol = 3, nrow = 2,
                 top = textGrob("Top Metabolitos", gp = gpar(fontsize = 20, fontface = "bold")))
}

# ─── Pareto ───────────────────────────────────────────────────────────────
if (nrow(pareto_res) > 0) {
  pareto_plots <- map(head(pareto_res$Feature, 6), ~plot_box_feat(.x, analysis, pareto_res)) %>%
    Filter(Negate(is.null), .)
  if (length(pareto_plots) > 0)
    grid.arrange(grobs = pareto_plots, ncol = 3, nrow = 2,
                 top = textGrob("Top Metricas Pareto", gp = gpar(fontsize = 20, fontface = "bold")))
}

# ─── Variables Clínicas ───────────────────────────────────────────────────
clinical_to_plot <- c("menopause_status.diagnoses", "primary_diagnosis.diagnoses",
                      "ER", "PR", "HER2", "Subtype", "age_at_index.demographic")
c_plots <- map(clinical_to_plot, ~plot_clinical(.x, analysis)) %>%
  Filter(Negate(is.null), .)

# ─── FIX 6: Verificar que hay plots antes de iterar ──────────────────────
if (length(c_plots) > 0) {
  for (i in seq(1, length(c_plots), by = 2)) {
    idx <- i:min(i + 1, length(c_plots))
    grid.arrange(grobs = c_plots[idx], nrow = length(idx), ncol = 1,
                 top = textGrob("Distribucion Clinica por Grupo",
                                gp = gpar(fontsize = 20, fontface = "bold")))
  }
}

# ─── Kaplan-Meier ─────────────────────────────────────────────────────────
# ─── FIX 7: Filtrar NAs y valores no positivos en OS.time ────────────────
if (all(c("OS.time", "OS") %in% colnames(analysis))) {
  km_data <- analysis %>%
    filter(!is.na(OS.time), !is.na(OS), OS.time > 0)

  if (nrow(km_data) > 0 && length(unique(km_data$Group)) == 2) {
    fit <- survfit(Surv(OS.time, OS) ~ Group, data = km_data)
    print(ggsurvplot(fit, data = km_data,
                     pval           = TRUE,
                     pval.size      = 5,
                     conf.int       = TRUE,
                     palette        = unname(group_colors),
                     risk.table     = TRUE,
                     risk.table.cex = 1.0,
                     surv.line.size = 1.2,
                     title          = "Curva de Supervivencia: Core vs Divergent",
                     xlab           = "Tiempo (dias)",
                     ylab           = "Probabilidad de Supervivencia",
                     legend.labs    = c("Core", "Divergent"),
                     theme          = base_theme +
                       theme(plot.title  = element_text(size = 22, face = "bold"),
                             axis.title  = element_text(size = 16),
                             axis.text   = element_text(size = 14),
                             legend.text = element_text(size = 14))))
  } else {
    warning("⚠️  Datos insuficientes para Kaplan-Meier tras filtrado.")
  }
}

dev.off()
cat("\n✅ Reporte generado: 'Reporte_Metabolismo_Clinico_2026.pdf'\n")

Warning message:
“Using `size` aesthetic for lines was deprecated in ggplot2 3.4.0.
ℹ Please use `linewidth` instead.
ℹ The deprecated feature was likely used in the ggpubr package.
  Please report the issue at <https://github.com/kassambara/ggpubr/issues>.”
Ignoring unknown labels:
• colour : "Strata"


pdf 
  2


✅ Reporte generado: 'Reporte_Metabolismo_Clinico_2026.pdf'


In [8]:
install.packages("pheatmap")

Installing package into ‘/Users/eduardoruiz/Library/R/arm64/4.4/library’
(as ‘lib’ is unspecified)




The downloaded binary packages are in
	/var/folders/x0/j78w4f8n0_z4l575_k16n2k40000gn/T//RtmpPiHpQZ/downloaded_packages
